# Week 3 Assignment

## Synthetic Data Generator with Multiple LLM Providers

This notebook implements a synthetic dataset generator powered by multiple Large Language Model (LLM) providers. The system generates structured customer support ticket data using models from OpenAI, Google Gemini, Anthropic, and DeepSeek.

The generator uses prompt engineering to produce diverse datasets under different styles (realistic, noisy, and edge cases), validates the output against a defined schema, and converts the results into a structured dataset. An interactive Gradio interface allows users to select the model provider, configure generation parameters, preview the generated data, and export it as a CSV file.

This approach demonstrates how modern LLMs can be used to quickly generate realistic synthetic datasets for experimentation, testing, and machine learning workflows.

In [ ]:
# imports
import os
import json
import random
import tempfile
from dotenv import load_dotenv
from datetime import datetime, timedelta
from typing import List, Literal
from openai import OpenAI
import gradio as gr
import pandas as pd
from pydantic import BaseModel, ValidationError

# --- Optional provider SDKs ---
# The app works with whichever API keys you provide.

# Load environment variables
load_dotenv(override=True)
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')


# urls
anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
deepseek_url = "https://api.deepseek.com"





anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
deepseek = OpenAI(api_key=deepseek_api_key, base_url=deepseek_url)
openai = OpenAI()

providers = {
    "OpenAI": openai,
    "Gemini": gemini,
    "DeepSeek": deepseek,
    "Anthropic": anthropic,
}

PROVIDER_MODELS = {
    "OpenAI": [
        "gpt-4.1",
        "gpt-4.1-mini",
        "gpt-4o",
        "gpt-4o-mini"
    ],
    "Gemini": [
        "gemini-1.5-pro",
        "gemini-1.5-flash",
        "gemini-2.0-flash"
    ],
    "Anthropic": [
        "claude-3-haiku-20240307",
        "claude-3-sonnet-20240229",
        "claude-3-opus-20240229"
    ],
    "DeepSeek": [
        "deepseek-chat",
        "deepseek-coder"
    ]
}

# -----------------------------
# Schema
# -----------------------------

Priority = Literal["low", "medium", "high", "urgent"]
Sentiment = Literal["calm", "frustrated", "angry", "confused"]
Status = Literal["open", "in_progress", "resolved", "closed"]
IssueCategory = Literal[
    "billing",
    "login",
    "shipping",
    "refund",
    "bug_report",
    "account_access",
    "subscription",
    "feature_request",
]
Product = Literal[
    "Web App",
    "Mobile App",
    "Analytics Suite",
    "Payments API",
    "CRM Portal",
]


class SupportTicket(BaseModel):
    ticket_id: str
    customer_name: str
    product: Product
    issue_category: IssueCategory
    priority: Priority
    sentiment: Sentiment
    created_at: str
    ticket_text: str
    status: Status


PRODUCTS = [
    "Web App",
    "Mobile App",
    "Analytics Suite",
    "Payments API",
    "CRM Portal",
]

ISSUE_CATEGORIES = [
    "billing",
    "login",
    "shipping",
    "refund",
    "bug_report",
    "account_access",
    "subscription",
    "feature_request",
]

PRIORITIES = ["low", "medium", "high", "urgent"]
SENTIMENTS = ["calm", "frustrated", "angry", "confused"]
STATUSES = ["open", "in_progress", "resolved", "closed"]


PROMPT_STYLES = {
    "realistic": "Generate realistic customer support tickets written naturally by normal users.",
    "noisy": "Generate tickets with typos, shorthand, messy punctuation, and incomplete descriptions.",
    "edge_case": "Generate rare, complex or unusual support issues that are hard to diagnose.",
}


# -----------------------------
# Utility
# -----------------------------


def make_ticket_id(i: int):
    return f"TKT-{datetime.utcnow().strftime('%Y%m%d')}-{i:04d}-{random.randint(100,999)}"


def prompt_builder(n_rows: int, style: str):

    return f"""
Generate {n_rows} synthetic customer support tickets.

Style:
{PROMPT_STYLES[style]}

Return JSON array only.

Each row must contain:
- ticket_id
- customer_name
- product
- issue_category
- priority
- sentiment
- created_at
- ticket_text
- status

Allowed values:
products = {PRODUCTS}
issue_category = {ISSUE_CATEGORIES}
priority = {PRIORITIES}
sentiment = {SENTIMENTS}
status = {STATUSES}

created_at format: YYYY-MM-DD HH:MM:SS

Make them diverse and realistic.
"""


# -----------------------------
# Provider generators
# -----------------------------


def generate(provider, model, prompt, temperature):
    client = providers.get(provider)

    res = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
    )

    return res.choices[0].message.content


# -----------------------------
# Parsing / validation
# -----------------------------

def parse_json(text):
    text = text.replace("```json", "").replace("```", "").strip()

    data = json.loads(text)

    valid = []

    for row in data:
        try:
            valid.append(SupportTicket(**row).model_dump())
        except ValidationError:
            pass

    return valid


# -----------------------------
# Main generation
# -----------------------------


def generate_dataset(provider, model, style, n_rows, temperature):
    prompt = prompt_builder(n_rows, style)
    raw = generate(provider, model, prompt, temperature)
    rows = parse_json(raw)
    df = pd.DataFrame(rows)
    path = os.path.join(
        tempfile.gettempdir(), f"synthetic_support_{int(datetime.utcnow().timestamp())}.csv"
    )

    df.to_csv(path, index=False)

    summary = f"Rows generated: {len(df)}"

    return df, summary, path


# -----------------------------
# UI
# -----------------------------

def update_models(provider):
    models = PROVIDER_MODELS.get(provider, [])
    return gr.Dropdown(choices=models, value=models[0])

with gr.Blocks() as demo:
    gr.Markdown("# Synthetic Data Generator")

    with gr.Row():
        provider = gr.Dropdown(
            ["OpenAI", "Gemini", "Anthropic", "DeepSeek"],
            value="OpenAI",
            label="Model Provider",
        )

        model = gr.Dropdown(
            choices=PROVIDER_MODELS["OpenAI"],
            value=PROVIDER_MODELS["OpenAI"][0],
            label="Model"
        )

        temperature = gr.Slider(
            0,
            1,
            value=0.9,
            step=0.1,
            label="Temperature"
        )

    with gr.Row():

        style = gr.Dropdown(
            ["realistic", "noisy", "edge_case"],
            value="realistic",
            label="Prompt Style",
        )

        rows = gr.Slider(10, 200, value=30, step=10, label="Rows")


    provider.change(
        fn=update_models,
        inputs=provider,
        outputs=model
    )

    btn = gr.Button("Generate Dataset")

    output_table = gr.Dataframe()

    summary = gr.Textbox(label="Summary")

    file_out = gr.File(label="Download CSV")

    btn.click(
        fn=generate_dataset,
        inputs=[provider, model, style, rows, temperature],
        outputs=[output_table, summary, file_out],
    )


demo.launch()
